In [1]:
import folium
from folium.plugins import TimestampedGeoJson

m = folium.Map(location=[35.68159659061569, 139.76451516151428], zoom_start=16)

# Lon, Lat order.
lines = [
    {
        "coordinates": [
            [139.76451516151428, 35.68159659061569],
            [139.75964426994324, 35.682590062684206],
        ],
        "dates": ["2017-06-02T00:00:00", "2017-06-02T00:10:00"],
        "color": "red",
    },
    {
        "coordinates": [
            [139.75964426994324, 35.682590062684206],
            [139.7575843334198, 35.679505030038506],
        ],
        "dates": ["2017-06-02T00:10:00", "2017-06-02T00:20:00"],
        "color": "blue",
    },
    {
        "coordinates": [
            [139.7575843334198, 35.679505030038506],
            [139.76337790489197, 35.678040905014065],
        ],
        "dates": ["2017-06-02T00:20:00", "2017-06-02T00:30:00"],
        "color": "green",
        "weight": 15,
    },
    {
        "coordinates": [
            [139.76337790489197, 35.678040905014065],
            [139.76451516151428, 35.68159659061569],
        ],
        "dates": ["2017-06-02T00:30:00", "2017-06-02T00:40:00"],
        "color": "#FFFFFF",
    },
]

features = [
    {
        "type": "Feature",
        "geometry": {
            "type": "LineString",
            "coordinates": line["coordinates"],
        },
        "properties": {
            "times": line["dates"],
            "title": "test",
            "style": {
                "color": line["color"],
                "weight": line["weight"] if "weight" in line else 5,
            },
        },
    }
    for line in lines
]

TimestampedGeoJson(
    {
        "type": "FeatureCollection",
        "features": features,
    },
    period="PT1M",
    add_last_point=True,
).add_to(m)

m.save("timestamped_map.html")

In [2]:
import folium
from folium.utilities import JsCode
from folium.features import GeoJsonPopup
from folium.plugins.timeline import Timeline, TimelineSlider
import requests

m = folium.Map()

data = requests.get(
    "https://raw.githubusercontent.com/python-visualization/folium-example-data/main/historical_country_borders.json"
).json()
data

{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'properties': {'name': 'Guyana',
    'start': -113688000000.0,
    'end': 1341028800000.0},
   'geometry': {'type': 'Polygon',
    'coordinates': [[[-58.1726178865394, 6.812218090163412],
      [-57.16236388559852, 6.056945089460001],
      [-57.13576388557375, 5.954100089364232],
      [-58.071399886445136, 4.15569108768932],
      [-58.04375488641939, 4.001527087545753],
      [-56.470635884954305, 1.944500085629997],
      [-56.52555488500545, 1.922500085609499],
      [-59.104726887407494, 1.344718085071406],
      [-59.24396388753718, 1.386527085110345],
      [-59.675281887938866, 4.373336087892028],
      [-59.67444588793809, 4.385136087903007],
      [-61.11610888928074, 5.63471808906678],
      [-61.38972688953557, 5.940000089351088],
      [-59.98111788822369, 8.518327091752354],
      [-59.99028188823223, 8.535273091768133],
      [-58.1726178865394, 6.812218090163412]]]}},
  {'type': 'Feature',
   'propertie

In [3]:
timeline = Timeline(
    data,
    style=JsCode("""
        function (data) {
            function getColorFor(str) {
                var hash = 0;
                for (var i = 0; i < str.length; i++) {
                    hash = str.charCodeAt(i) + ((hash << 5) - hash);
                }
                var red = (hash >> 24) & 0xff;
                var grn = (hash >> 16) & 0xff;
                var blu = (hash >> 8) & 0xff;
                return "rgb(" + red + "," + grn + "," + blu + ")";
            }
            return {
                stroke: false,
                color: getColorFor(data.properties.name),
                fillOpacity: 0.5,
            };
        }
    """)
).add_to(m)
GeoJsonPopup(fields=['name'], labels=True).add_to(timeline)
TimelineSlider(
    auto_play=False,
    show_ticks=True,
    enable_keyboard_controls=True,
    playback_duration=100,
).add_timelines(timeline).add_to(m)

m.save("timeline_map.html")